# Step 5: RAG (Retrieval-Augmented Generation)
**AI Agents From Scratch — `ai-agents-from-scratch`**

Goals:
- Load and chunk a PDF document
- Embed chunks using `sentence-transformers` (free, no API key)
- Store and search embeddings using `ChromaDB`
- Pass retrieved context to Groq LLM to generate grounded answers

Stack: `pypdf` · `sentence-transformers` · `chromadb` · `groq`

In [1]:
# Install dependencies (run once)
!pip install pypdf sentence-transformers chromadb groq -q

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] T

## 1. Load and Chunk the PDF

LLMs have a context window limit, so we split the PDF into small overlapping chunks.
Each chunk is embedded and stored separately.

In [2]:
from pypdf import PdfReader

def load_pdf(path):
    """Extract all text from a PDF file."""
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks.
    chunk_size : characters per chunk
    overlap    : characters shared between consecutive chunks
                 (helps avoid cutting a sentence mid-thought)
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if c]  # remove empty chunks

# ── Load the sample PDF ──────────────────────────────────────────────────────
# Upload your own PDF to Colab and change this path, or use the sample below.
PDF_PATH = "ai_agents_guide.pdf"   # <-- change if using your own PDF

raw_text = load_pdf(PDF_PATH)
chunks   = chunk_text(raw_text)

print(f"Total characters : {len(raw_text)}")
print(f"Total chunks     : {len(chunks)}")
print(f"\n--- Sample chunk ---\n{chunks[0]}")

Total characters : 2268
Total chunks     : 6

--- Sample chunk ---
Introduction to AI Agents
What is an LLM?
A Large Language Model (LLM) is a type of artificial intelligence trained on massive amounts of text data.
LLMs can understand and generate human language. Popular examples include GPT-4, Claude, and LLaMA.
They are used in chatbots, code generation, summarization, and question answering.
What is RAG?
RAG stands for Retrieval-Augmented Generation. It is a technique that combines a retrieval system with a
language model. Instead of relying only on the mod


## 2. Embed Chunks and Store in ChromaDB

ChromaDB is a local vector database — it saves embeddings to disk so you don't
re-embed the same document every time.

In [4]:
import chromadb
from chromadb.utils import embedding_functions

# Use the same free model from Step 4
EMBED_MODEL = "all-MiniLM-L6-v2"

embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBED_MODEL
)

# Persistent client — saves to ./chroma_db folder
client     = chromadb.PersistentClient(path="./chroma_db")

# Delete collection if it already exists (clean slate on re-run)
try:
    client.delete_collection("rag_docs")
except:
    pass

collection = client.create_collection(
    name="rag_docs",
    embedding_function=embed_fn
)

# Index all chunks
print("Indexing chunks into ChromaDB...")
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"Done. {collection.count()} chunks stored.")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\zaras\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 3. Retrieval — Find Relevant Chunks for a Query

ChromaDB embeds the query automatically and returns the closest chunks.
This replaces the manual `cosine_similarity` loop from Step 4.

In [ ]:
def retrieve(query, top_k=3):
    """Return the top_k most relevant chunks for a query."""
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    return results["documents"][0]  # list of chunk strings

# Test retrieval
query = "What is RAG and how does it work?"
top_chunks = retrieve(query)

print(f'Query: "{query}"\n')
for i, chunk in enumerate(top_chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()

## 4. Generation — Pass Retrieved Context to Groq LLM

This is the **Augmented Generation** part of RAG.
We inject the retrieved chunks into the prompt so the LLM answers from
the document, not just its training data.

In [ ]:
from groq import Groq

GROQ_API_KEY = "gsk_your_key_here"   # <-- paste your Groq API key
groq_client  = Groq(api_key=GROQ_API_KEY)

def rag_answer(query, top_k=3):
    """
    Full RAG pipeline:
      1. Retrieve top_k relevant chunks from ChromaDB
      2. Build a prompt with context + query
      3. Send to Groq LLM and return the answer
    """
    # Step 1 — Retrieve
    chunks = retrieve(query, top_k=top_k)
    context = "\n\n".join(chunks)

    # Step 2 — Build prompt
    system_prompt = (
        "You are a helpful assistant. Answer the user's question using ONLY "
        "the context provided below. If the answer is not in the context, "
        "say 'I don't know based on the provided document.'\n\n"
        f"Context:\n{context}"
    )

    # Step 3 — Generate
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": query}
        ]
    )
    return response.choices[0].message.content

print("RAG pipeline ready.")

## 5. Ask Questions About the PDF

In [ ]:
questions = [
    "What is RAG and why is it useful?",
    "How do embeddings work?",
    "What is an AI agent?",
    "What is ChromaDB used for?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_answer(q)}")
    print("-" * 60)

## 6. Try Your Own Question

In [ ]:
your_question = "What is tool calling?"   # <-- change this
print(rag_answer(your_question))

## 7. RAG Pipeline — How It All Connects

```
PDF File
   │
   ▼
load_pdf()  →  chunk_text()  →  chunks[]
                                    │
                              ChromaDB.add()   ← sentence-transformers embeds each chunk
                                    │
                         [Stored on disk as vectors]

                                    │  (at query time)
                                    ▼
User Query  →  ChromaDB.query()  →  Top-K chunks
                                    │
                              System prompt + chunks
                                    │
                              Groq LLM (llama-3.3-70b)
                                    │
                                    ▼
                             Grounded Answer
```

---
## Summary

| Step | What happened |
|------|---------------|
| Load | PDF text extracted with `pypdf` |
| Chunk | Text split into 500-char overlapping pieces |
| Embed | Each chunk embedded by `all-MiniLM-L6-v2` |
| Store | Embeddings saved to ChromaDB (persisted on disk) |
| Retrieve | Query embedded, top-3 chunks fetched by similarity |
| Generate | Chunks + query sent to Groq → grounded answer |

**Next:** Step 6 — Memory (giving your agent short-term and long-term memory)